**Imports**

In [ ]:
import hashlib
import json
import math
from dataclasses import dataclass
from pathlib import Path
from typing import Callable, Iterable, Mapping

import yaml

A **cycle** is the time one bus needs to leave its starting terminal, drive the whole route out and back (stopping at each stop on the way), take its layover, and be ready to start the same trip again. We take the full round trip, not one direction, because a bus can only start its next trip after it gets back. The number of buses a route needs is therefore set by the full round trip, `vehicles = ceil(cycle / headway)`. The two directions can also have different lengths (one-way streets) and different stop counts, so doubling one direction would be wrong.


**1. Dwell is in seconds, the cycle is in minutes. Where does the conversion happen?**
Inside `cycle_min`, at one point only: `dwell_s / 60`. The caller passes dwell in its natural unit (seconds, as in the assumption sheet: 20 s), and the function returns minutes. The parameter names carry the unit (`_s`, `_min`, `_km`, `_kmh`), so mixing them up is visible.

**2. What should happen if the speed is 0 or negative?**
Raise a `ValueError`. A speed of 0 would mean infinite run time and a negative speed has no meaning. Returning `inf`, `0`, or a clamped value would hide a data error and go on to give a wrong vehicle count. That breaks the rule *fail loudly, do not guess*. The same applies to negative lengths, dwell, layover or stop counts, and to non-finite values.

**3. If only half of the route is known (partial coverage), should this function still return a number?**
No. This function does not decide that. It is pure arithmetic: it trusts that the inputs describe the **full** round trip. Whether a route is fully covered by curated stops and drive paths is decided earlier, where the route length is built (the sum of drive paths between consecutive curated stops). That step must refuse to produce a length for a partially covered route, so `cycle_min` is never called with half a route.


**Cycle time for one full round trip**

In [2]:
def cycle_min(
    route_length_km: float,
    speed_kmh: float,
    stops_per_cycle: int,
    dwell_s: float,
    layover_min: float,
) -> float:
    """Full round-trip cycle time in minutes.

    cycle = route_length_km / speed_kmh * 60 + stops_per_cycle * dwell_s / 60 + layover_min

    route_length_km: both directions together (sum of drive paths between curated stops).
    stops_per_cycle: stop events over both directions.
    layover_min: total layover per cycle.
    """
    values = {
        "route_length_km": route_length_km,
        "speed_kmh": speed_kmh,
        "stops_per_cycle": stops_per_cycle,
        "dwell_s": dwell_s,
        "layover_min": layover_min,
    }
    for name, value in values.items():
        if not math.isfinite(value):
            raise ValueError(f"{name} must be finite, got {value}")
        if value < 0:
            raise ValueError(f"{name} must be >= 0, got {value}")
    if speed_kmh == 0:
        raise ValueError("speed_kmh must be > 0, got 0")
    if route_length_km == 0:
        raise ValueError("route_length_km must be > 0, got 0")

    run_min = route_length_km / speed_kmh * 60
    dwell_min = stops_per_cycle * dwell_s / 60
    return run_min + dwell_min + layover_min


**cycle_min quick check**

In [3]:
# 12 km round trip at 15 km/h, 30 stop events x 20 s, 5 min layover
# run 48 + dwell 10 + layover 5 = 63 min
assert math.isclose(cycle_min(12, 15, 30, 20, 5), 63.0)

for bad_speed in (0, -10):
    try:
        cycle_min(12, bad_speed, 30, 20, 5)
    except ValueError as e:
        print("rejected:", e)
    else:
        raise AssertionError("bad speed accepted")

cycle_min(12, 15, 30, 20, 5)


rejected: speed_kmh must be > 0, got 0
rejected: speed_kmh must be >= 0, got -10


63.0

**cycle_min tests**

In [4]:
# --- cycle_min tests (expected values computed by hand) ---

def check(expected, **kw):
    got = cycle_min(**kw)
    assert math.isclose(got, expected, abs_tol=1e-9), f"{kw} -> {got}, expected {expected}"

def must_fail(**kw):
    try:
        cycle_min(**kw)
    except ValueError:
        return
    raise AssertionError(f"bad input accepted: {kw}")

# 12 km round trip at 15 km/h, 30 stop events x 20 s, 5 min layover
base = dict(route_length_km=12, speed_kmh=15, stops_per_cycle=30, dwell_s=20, layover_min=5)

# 1. Normal case: run 48 + dwell 10 + layover 5 = 63
check(63.0, **base)

# 2. Each part checked on its own (catches a term that is missing or counted twice)
check(53.0, **{**base, "dwell_s": 0})                              # 48 + 0 + 5
check(58.0, **{**base, "layover_min": 0})                          # 48 + 10 + 0
check(48.0, **{**base, "stops_per_cycle": 0, "layover_min": 0})    # run time only

# 3. Lower speed -> longer cycle (the stress-test direction): 60 + 10 + 5 = 75
check(75.0, **{**base, "speed_kmh": 12})

# 4. Seconds-to-minutes conversion: 6 stops x 30 s = 3 min
check(48.0 + 3.0, **{**base, "stops_per_cycle": 6, "dwell_s": 30, "layover_min": 0})

# 5. Invalid inputs must fail loudly
for name, bad in [
    ("speed_kmh", 0), ("speed_kmh", -10),
    ("route_length_km", 0), ("route_length_km", -1),
    ("dwell_s", -1), ("layover_min", -1), ("stops_per_cycle", -1),
    ("speed_kmh", float("nan")), ("route_length_km", float("inf")),
]:
    must_fail(**{**base, name: bad})

print("cycle_min: all tests passed")

cycle_min: all tests passed


**Vehicles needed for a headway**

In [ ]:
VEHICLE_RATIO_TOL = 1e-9


def vehicles(cycle_min: float, headway_min: float) -> int:
    """Vehicles needed to run one cycle at a headway: ceil(cycle / headway).

    A ratio within 1e-9 of an integer is float noise and uses that integer.
    A headway longer than the cycle needs one vehicle.
    """
    for name, value in (("cycle_min", cycle_min), ("headway_min", headway_min)):
        if not math.isfinite(value):
            raise ValueError(f"{name} must be finite, got {value}")
        if value <= 0:
            raise ValueError(f"{name} must be > 0, got {value}")
    if headway_min > cycle_min:
        return 1
    ratio = cycle_min / headway_min
    nearest = round(ratio)
    if abs(ratio - nearest) <= VEHICLE_RATIO_TOL:
        return int(nearest)
    return math.ceil(ratio)

**Model catalog (stub)**

In [ ]:
COVERAGE_VALUES = ("full", "partial")
CYCLE_FIELDS = ("full_cycle_km", "speed_kmh", "stops_per_cycle", "dwell_s", "layover_min")
LINK_KINDS = ("walk", "bike")


@dataclass(frozen=True)
class ServiceGroup:
    """Existing service group or pre-built service. Cycle facts may be None only when coverage is partial."""
    service_group_id: str
    pattern_ids: tuple[str, ...]
    base_headway_min: float
    full_cycle_km: float | None
    speed_kmh: float | None
    stops_per_cycle: int | None
    dwell_s: float | None
    layover_min: float | None
    coverage: str
    status: str
    speed_status: str

    def __post_init__(self):
        if self.coverage not in COVERAGE_VALUES:
            raise ValueError(f"{self.service_group_id}: coverage must be one of {COVERAGE_VALUES}, got {self.coverage!r}")
        if self.coverage == "full":
            missing = [name for name in CYCLE_FIELDS if getattr(self, name) is None]
            if missing:
                raise ValueError(f"{self.service_group_id}: full coverage but missing {missing}")


@dataclass(frozen=True)
class Link:
    link_id: str
    kind: str
    length_m: float
    requires_construction: bool
    status: str

    def __post_init__(self):
        if self.kind not in LINK_KINDS:
            raise ValueError(f"{self.link_id}: kind must be one of {LINK_KINDS}, got {self.kind!r}")
        if not math.isfinite(self.length_m) or self.length_m < 0:
            raise ValueError(f"{self.link_id}: length_m must be finite and >= 0, got {self.length_m}")


@dataclass(frozen=True)
class Parking:
    parking_id: str
    status: str


@dataclass(frozen=True)
class ModelCatalog:
    """Stub of Celal's ModelCatalog with the fields compute_resources needs."""
    service_groups: Mapping[str, ServiceGroup]
    prebuilt_services: Mapping[str, ServiceGroup]
    links: Mapping[str, Link]
    parking: Mapping[str, Parking]

**Intervention candidate**

In [ ]:
CANDIDATE_KINDS = ("headway_cut", "activate_service", "link_bundle")


@dataclass(frozen=True)
class Candidate:
    candidate_id: str
    kind: str
    target_id: str | None = None
    pct: float | None = None
    link_ids: tuple[str, ...] = ()
    parking_ids: tuple[str, ...] = ()

    def __post_init__(self):
        cid = self.candidate_id
        if self.kind not in CANDIDATE_KINDS:
            raise ValueError(f"{cid}: kind must be one of {CANDIDATE_KINDS}, got {self.kind!r}")
        if self.kind == "link_bundle":
            if self.target_id is not None or self.pct is not None:
                raise ValueError(f"{cid}: link_bundle takes no target_id or pct")
            if not self.link_ids and not self.parking_ids:
                raise ValueError(f"{cid}: link_bundle needs link_ids or parking_ids")
            return
        if self.target_id is None:
            raise ValueError(f"{cid}: {self.kind} needs target_id")
        if self.link_ids or self.parking_ids:
            raise ValueError(f"{cid}: {self.kind} takes no link_ids or parking_ids")
        if self.kind == "activate_service":
            if self.pct is not None:
                raise ValueError(f"{cid}: activate_service takes no pct")
            return
        if self.pct is None or not math.isfinite(self.pct) or not 0 < self.pct < 1:
            raise ValueError(f"{cid}: headway_cut pct must be in (0, 1), got {self.pct}")

**Scenario with analysis hours from limits.yaml**

In [ ]:
@dataclass(frozen=True)
class Scenario:
    speed_factor: float
    headway_factor: float
    H_hours: float

    def __post_init__(self):
        for name in ("speed_factor", "headway_factor", "H_hours"):
            value = getattr(self, name)
            if value is None:
                raise ValueError(f"{name} is missing")
            if isinstance(value, bool):
                raise ValueError(f"{name} must be a number, got bool {value}")
            if not math.isfinite(value) or value <= 0:
                raise ValueError(f"{name} must be finite and > 0, got {value}")


def load_scenario(limits_path: str | Path, speed_factor: float, headway_factor: float) -> Scenario:
    """Scenario with H_hours taken from config/limits.yaml."""
    with open(limits_path) as f:
        limits = yaml.safe_load(f) or {}
    if "H_hours" not in limits:
        raise ValueError(f"H_hours missing in {limits_path}")
    return Scenario(speed_factor=speed_factor, headway_factor=headway_factor, H_hours=limits["H_hours"])

**Resource result type**

In [ ]:
@dataclass(frozen=True)
class ServiceResource:
    service_id: str
    candidate_id: str
    base_headway_min: float | None
    new_headway_min: float
    base_cycle_min: float | None
    new_cycle_min: float | None
    base_vehicles: int | None
    new_vehicles: int | None
    delta_vehicles: int | None
    status: str
    warnings: tuple[str, ...]


@dataclass(frozen=True)
class ResourceResult:
    added_bus_hours: float | None
    known_part_bus_hours: float
    new_infra_km: float
    action_count: int
    status: str
    warnings: tuple[str, ...]
    per_service: tuple[ServiceResource, ...]

**Resources for one package**

In [ ]:
M_PER_KM = 1000


def _unique_candidates(package: Iterable[Candidate]) -> list[Candidate]:
    by_id: dict[str, Candidate] = {}
    for c in package:
        if c.candidate_id in by_id and by_id[c.candidate_id] != c:
            raise ValueError(f"{c.candidate_id}: two different definitions in one package")
        by_id[c.candidate_id] = c
    return [by_id[cid] for cid in sorted(by_id)]


def _cycle(service: ServiceGroup, scenario: Scenario) -> float:
    return cycle_min(
        service.full_cycle_km,
        service.speed_kmh * scenario.speed_factor,
        service.stops_per_cycle,
        service.dwell_s,
        service.layover_min,
    )


def _headway_cut(cut: Candidate, group: ServiceGroup, scenario: Scenario) -> ServiceResource:
    base_headway = group.base_headway_min * scenario.headway_factor
    new_headway = base_headway * (1 - cut.pct)
    if group.coverage == "partial":
        return ServiceResource(group.service_group_id, cut.candidate_id, base_headway, new_headway,
                               None, None, None, None, None, "unknown", ())
    cycle = _cycle(group, scenario)
    base_v = vehicles(cycle, base_headway)
    new_v = vehicles(cycle, new_headway)
    delta = new_v - base_v
    if delta < 0:
        raise ValueError(f"{cut.candidate_id}: negative vehicle delta {delta} on {group.service_group_id}")
    warnings = []
    if base_headway > cycle or new_headway > cycle:
        warnings.append("headway_exceeds_cycle")
    if delta == 0:
        warnings.append("zero_cost_uses_slack")
    return ServiceResource(group.service_group_id, cut.candidate_id, base_headway, new_headway,
                           cycle, cycle, base_v, new_v, delta, "known", tuple(warnings))


def _activation(act: Candidate, service: ServiceGroup, scenario: Scenario) -> ServiceResource:
    # Pre-built service headway is a design value; headway_factor applies to existing base headways only
    headway = service.base_headway_min
    if service.coverage == "partial":
        return ServiceResource(service.service_group_id, act.candidate_id, None, headway,
                               None, None, 0, None, None, "unknown", ())
    cycle = _cycle(service, scenario)
    new_v = vehicles(cycle, headway)
    warnings = ("headway_exceeds_cycle",) if headway > cycle else ()
    return ServiceResource(service.service_group_id, act.candidate_id, None, headway,
                           None, cycle, 0, new_v, new_v, "known", warnings)


def compute_resources(package: Iterable[Candidate], catalog: ModelCatalog, scenario: Scenario) -> ResourceResult:
    """Resources one package needs under one scenario. Does not compare against limits."""
    candidates = _unique_candidates(package)
    cuts: dict[str, Candidate] = {}
    activations: dict[str, Candidate] = {}
    link_ids: set[str] = set()

    for c in candidates:
        if c.kind == "headway_cut":
            if c.target_id not in catalog.service_groups:
                raise ValueError(f"{c.candidate_id}: unknown service group {c.target_id!r}")
            if c.target_id in cuts:
                raise ValueError(f"two headway actions on {c.target_id}: {cuts[c.target_id].candidate_id}, {c.candidate_id}")
            cuts[c.target_id] = c
        elif c.kind == "activate_service":
            if c.target_id not in catalog.prebuilt_services:
                raise ValueError(f"{c.candidate_id}: unknown pre-built service {c.target_id!r}")
            if c.target_id in activations:
                raise ValueError(f"two activations of {c.target_id}: {activations[c.target_id].candidate_id}, {c.candidate_id}")
            activations[c.target_id] = c
        else:
            for link_id in c.link_ids:
                if link_id not in catalog.links:
                    raise ValueError(f"{c.candidate_id}: unknown link {link_id!r}")
                link_ids.add(link_id)
            for parking_id in c.parking_ids:
                if parking_id not in catalog.parking:
                    raise ValueError(f"{c.candidate_id}: unknown parking {parking_id!r}")

    computed = [_headway_cut(cut, catalog.service_groups[sid], scenario) for sid, cut in cuts.items()]
    computed += [_activation(act, catalog.prebuilt_services[sid], scenario) for sid, act in activations.items()]
    per_service = tuple(sorted(computed, key=lambda s: s.service_id))
    warnings = tuple(dict.fromkeys(w for s in per_service for w in s.warnings))
    known_part = sum(s.delta_vehicles for s in per_service if s.status == "known") * scenario.H_hours
    status = "unknown" if any(s.status == "unknown" for s in per_service) else "known"
    construction_m = sum(catalog.links[lid].length_m for lid in link_ids if catalog.links[lid].requires_construction)

    return ResourceResult(
        added_bus_hours=None if status == "unknown" else known_part,
        known_part_bus_hours=known_part,
        new_infra_km=construction_m / M_PER_KM,
        action_count=len(candidates),
        status=status,
        warnings=warnings,
        per_service=per_service,
    )

**Routing request and evaluation result (stubs)**

In [ ]:
# STUB: replace with Celal's contracts (RoutingRequest, EvaluationResult) when they land
OD_STATUSES = ("reachable", "over_cap", "no_path")


@dataclass(frozen=True)
class RoutingRequest:
    """Stub. Package ids, accessibility thresholds T and time cap C, all in minutes."""
    candidate_ids: tuple[str, ...]
    thresholds_min: tuple[int, ...]
    primary_t_min: int
    cap_min: float

    def __post_init__(self):
        if self.primary_t_min not in self.thresholds_min:
            raise ValueError(f"primary_t_min {self.primary_t_min} not in thresholds {self.thresholds_min}")
        if isinstance(self.cap_min, bool) or not math.isfinite(self.cap_min) or self.cap_min <= 0:
            raise ValueError(f"cap_min must be finite and > 0, got {self.cap_min}")


@dataclass(frozen=True)
class ODResult:
    """Stub. Unreachable ODs carry minutes = C and status no_path, never NaN or inf."""
    od_id: str
    minutes: float
    status: str

    def __post_init__(self):
        if self.status not in OD_STATUSES:
            raise ValueError(f"{self.od_id}: status must be one of {OD_STATUSES}, got {self.status!r}")
        if not math.isfinite(self.minutes):
            raise ValueError(f"{self.od_id}: minutes must be finite, got {self.minutes}")


@dataclass(frozen=True)
class EvaluationResult:
    """Stub. within_counts maps each threshold T to the number of ODs reached within T."""
    config_hash: str
    candidate_ids: tuple[str, ...]
    od_results: tuple[ODResult, ...]
    within_counts: Mapping[int, int]
    primary_t_min: int
    capped_time_sum: float

**Fixture evaluator**

In [ ]:
FIXTURE_LABEL = "synthetic test fixture, not a city result"


def load_fixture(path: str | Path) -> dict:
    with open(path) as f:
        fixture = json.load(f)
    if fixture.get("label") != FIXTURE_LABEL:
        raise ValueError(f"{path}: missing fixture label {FIXTURE_LABEL!r}")
    return fixture


def make_fixture_evaluator(fixture: Mapping) -> Callable[[RoutingRequest], EvaluationResult]:
    """evaluator(RoutingRequest) -> EvaluationResult over a synthetic fixture.

    OD time = min(disrupted time, every option whose required actions are all in the package).
    """
    action_ids = set(fixture["actions"])
    config_hash = hashlib.sha256(json.dumps(fixture, sort_keys=True).encode()).hexdigest()[:12]

    def evaluate(request: RoutingRequest) -> EvaluationResult:
        package = set(request.candidate_ids)
        unknown = package - action_ids
        if unknown:
            raise ValueError(f"unknown candidate ids {sorted(unknown)}")
        od_results = []
        for od in fixture["ods"]:
            times = [] if od["disrupted_min"] is None else [od["disrupted_min"]]
            times += [opt["minutes"] for opt in od["options"] if set(opt["requires"]) <= package]
            if not times:
                od_results.append(ODResult(od["od_id"], request.cap_min, "no_path"))
                continue
            t = min(times)
            od_results.append(ODResult(od["od_id"], t, "over_cap" if t > request.cap_min else "reachable"))
        within = {
            t_min: sum(1 for r in od_results if r.status != "no_path" and r.minutes <= t_min)
            for t_min in request.thresholds_min
        }
        capped = sum(min(r.minutes, request.cap_min) for r in od_results)
        return EvaluationResult(config_hash, tuple(sorted(package)), tuple(od_results),
                                within, request.primary_t_min, capped)

    return evaluate

**Ranking key for packages**

In [ ]:
def rank_key(result: EvaluationResult, resources: ResourceResult) -> tuple:
    """Sort key, best first: more ODs within T, lower capped sum, fewer bus-hours,
    fewer new km, fewer actions, then candidate ids alphabetically."""
    if resources.added_bus_hours is None:
        raise ValueError(f"cannot rank {result.candidate_ids}: bus-hours unknown")
    return (
        -result.within_counts[result.primary_t_min],
        result.capped_time_sum,
        resources.added_bus_hours,
        resources.new_infra_km,
        resources.action_count,
        tuple(sorted(result.candidate_ids)),
    )

**Fixture catalog and candidates**

In [ ]:
FIXTURE = "fixture"


def fixture_service(sid, headway, km, speed, stops, dwell, layover, coverage="full"):
    return ServiceGroup(sid, (f"{sid}_out", f"{sid}_in"), headway, km, speed, stops, dwell, layover,
                        coverage, FIXTURE, FIXTURE)


catalog = ModelCatalog(
    service_groups={
        "SG_R2": fixture_service("SG_R2", 10, 12, 15, 30, 20, 5),
        "SG_R14": fixture_service("SG_R14", 12, None, None, None, None, None, coverage="partial"),
    },
    prebuilt_services={
        "SVC_SHUTTLE": fixture_service("SVC_SHUTTLE", 8, 9, 18, 6, 30, 4),
        "SVC_LOOP": fixture_service("SVC_LOOP", 20, 4, 20, 4, 30, 3),
    },
    links={
        "L1": Link("L1", "walk", 300, False, FIXTURE),
        "L2": Link("L2", "bike", 1200, True, FIXTURE),
        "L3": Link("L3", "walk", 40, True, FIXTURE),
    },
    parking={"P1": Parking("P1", FIXTURE)},
)

cands = {
    "a1": Candidate("a1", "headway_cut", target_id="SG_R2", pct=0.20),
    "a1b": Candidate("a1b", "headway_cut", target_id="SG_R2", pct=0.30),
    "a8": Candidate("a8", "headway_cut", target_id="SG_R2", pct=0.05),
    "a2": Candidate("a2", "headway_cut", target_id="SG_R14", pct=0.20),
    "a3": Candidate("a3", "activate_service", target_id="SVC_SHUTTLE"),
    "a3b": Candidate("a3b", "activate_service", target_id="SVC_SHUTTLE"),
    "a9": Candidate("a9", "activate_service", target_id="SVC_LOOP"),
    "a45": Candidate("a45", "link_bundle", link_ids=("L2",), parking_ids=("P1",)),
    "a6": Candidate("a6", "link_bundle", link_ids=("L1",)),
    "a7": Candidate("a7", "link_bundle", link_ids=("L2", "L3")),
}

base_scenario = load_scenario(Path("../config/limits.yaml"), speed_factor=1.0, headway_factor=1.0)


def res(*ids, scenario=base_scenario):
    return compute_resources([cands[i] for i in ids], catalog, scenario)


def veh(result, sid):
    s = next(s for s in result.per_service if s.service_id == sid)
    return s.base_vehicles, s.new_vehicles

**compute_resources tests**

In [ ]:
# Expected values computed by hand
r = res("a1");   assert (r.added_bus_hours, veh(r, "SG_R2"), r.new_infra_km, r.action_count, r.status, r.warnings) == (3, (7, 8), 0, 1, "known", ())  # E1
r = res("a1b");  assert (r.added_bus_hours, veh(r, "SG_R2")) == (6, (7, 9))                                     # E2
r = res("a8");   assert (r.added_bus_hours, veh(r, "SG_R2"), r.warnings) == (0, (7, 7), ("zero_cost_uses_slack",))  # E3
assert r.per_service[0].warnings == ("zero_cost_uses_slack",)
r = res("a3");   assert (r.added_bus_hours, veh(r, "SVC_SHUTTLE"), r.status) == (15, (0, 5), "known")           # E4
r = res("a9");   assert (r.added_bus_hours, veh(r, "SVC_LOOP"), r.warnings) == (3, (0, 1), ("headway_exceeds_cycle",))  # E5
assert r.per_service[0].warnings == ("headway_exceeds_cycle",)
r = res("a1", "a3", "a45")
assert r.added_bus_hours == 18 and math.isclose(r.new_infra_km, 1.2) and r.action_count == 3 and r.status == "known"  # E6
r = res("a45", "a7"); assert math.isclose(r.new_infra_km, 1.24) and r.action_count == 2                         # E7
r = res("a6");   assert (r.new_infra_km, r.action_count) == (0, 1)                                              # E8
r = res("a1", "a2"); assert (r.status, r.added_bus_hours, r.known_part_bus_hours) == ("unknown", None, 3)       # E9

try:                                                                                                             # E10
    res("a1", "a1b")
    raise AssertionError("E10: two cuts on one group accepted")
except ValueError as e:
    assert "a1" in str(e) and "a1b" in str(e)

try:                                                                                                             # same service activated twice
    res("a3", "a3b")
    raise AssertionError("two activations of one service accepted")
except ValueError as e:
    assert "a3" in str(e) and "a3b" in str(e)

r = res("a1", scenario=Scenario(0.8, 1.0, 3.0))                                                                 # E11
assert math.isclose(r.per_service[0].new_cycle_min, 75) and veh(r, "SG_R2") == (8, 10) and r.added_bus_hours == 6
r = res("a1", scenario=Scenario(1.0, 0.8, 3.0))                                                                 # E12
assert math.isclose(r.per_service[0].new_headway_min, 6.4) and veh(r, "SG_R2") == (8, 10) and r.added_bus_hours == 6
assert res("a3", "a1") == res("a1", "a3")                                                                       # E13

for bad in (lambda: Scenario(1.0, 1.0), lambda: Scenario(1.0, 1.0, None),
            lambda: Scenario(True, 1.0, 3.0), lambda: Scenario(1.0, True, 3.0), lambda: Scenario(1.0, 1.0, True)):                                      # E14
    try:
        bad()
        raise AssertionError("E14: scenario without H accepted")
    except (TypeError, ValueError):
        pass

assert vehicles(66.00000000000001, 6) == 11 and vehicles(66.06, 6) == 12 and vehicles(63, 10) == 7             # E15
for bad in (lambda: vehicles(63, 0), lambda: vehicles(63, -5),                                                  # E16
            *(lambda p=p: Candidate("bad", "headway_cut", target_id="SG_R2", pct=p) for p in (0, -0.1, 1, 1.5))):  # E17
    try:
        bad()
        raise AssertionError("E16/E17: bad input accepted")
    except ValueError:
        pass

print("compute_resources: all tests passed")

**Four-action fixture**

In [ ]:
fx = load_fixture(Path("../tests/fixtures/decision/fixture_4action.json"))
fx_evaluate = make_fixture_evaluator(fx)
fx_limits = fx["limits"]


def fx_request(ids):
    thresholds = tuple(sorted([fx_limits["T_min"], *fx_limits["T_sensitivity_min"]]))
    return RoutingRequest(tuple(ids), thresholds, fx_limits["T_min"], fx_limits["C_min"])


def fx_resources(ids):
    bus_hours = sum(fx["actions"][i]["bus_hours"] for i in ids)
    km = sum(fx["actions"][i]["new_infra_km"] for i in ids)
    return ResourceResult(bus_hours, bus_hours, km, len(ids), "known", (), ())

**Fixture evaluator and ranking tests**

In [ ]:
# Expected values computed by hand: n40 / n45 / n50 / capped sum
expected = {
    "": (0, 0, 2, 507), "A": (1, 2, 4, 467), "B": (2, 4, 6, 437), "C": (1, 3, 5, 398), "D": (0, 0, 2, 507),
    "AC": (2, 5, 7, 358), "AD": (1, 2, 4, 467), "BD": (2, 4, 6, 437), "CD": (1, 5, 5, 390), "ACD": (2, 7, 7, 350),
}
for pkg, (n40, n45, n50, capped) in expected.items():
    r = fx_evaluate(fx_request(pkg))
    assert (r.within_counts[40], r.within_counts[45], r.within_counts[50], r.capped_time_sum) == (n40, n45, n50, capped), pkg

assert [od.minutes for od in fx_evaluate(fx_request("ACD")).od_results] == [40, 42, 55, 41, 39, 44, 44, 45]
od5 = fx_evaluate(fx_request("B")).od_results[4]
assert (od5.od_id, od5.minutes, od5.status) == ("OD5", 120, "no_path")

ranked = sorted(expected, key=lambda p: rank_key(fx_evaluate(fx_request(p)), fx_resources(p)))
assert ranked == ["ACD", "AC", "CD", "B", "BD", "C", "A", "AD", "", "D"], ranked

synthetic = EvaluationResult("synthetic", ("X",), (), {45: 5}, 45, 300)
assert rank_key(synthetic, ResourceResult(6, 6, 0, 1, "known", (), ())) < rank_key(synthetic, ResourceResult(9, 9, 0, 1, "known", (), ()))
assert rank_key(synthetic, ResourceResult(6, 6, 0.5, 1, "known", (), ())) < rank_key(synthetic, ResourceResult(6, 6, 0.5, 2, "known", (), ()))

print("fixture evaluator and rank_key: all tests passed")